# N-gram model

Count-based n-gram over `data/processed/names.txt` (no training: the model is
the smoothed count tensor). The logic lives in the package — `onomatron.data`
(names, vocab, n-gram pairs) and `onomatron.models.ngram.NGramModel` — this notebook
only wires it up, plots the bigram count matrix and samples names. n=2 is the
plain bigram model.


In [1]:
import plotly.express as px

from onomatron.data import Vocab, load_names, split_names
from onomatron.models.ngram import NGramModel

## Load names, split train/val/test, build the vocab

The `'.'` sentinel is index 0 and marks both the beginning and the end of a
name — safe because the pipeline guarantees names.txt holds only latin letters
and spaces.

The split is done on names (never on n-grams, which are correlated within a
name), with a fixed seed so every notebook works on the same splits. The vocab
is built on train only, so no information leaks from val/test.

In [2]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train"
    f" / {len(val_names)} val / {len(test_names)} test"
)
print(f"vocab of {len(vocab)} chars")

6745 names -> 5396 train / 674 val / 675 test
vocab of 38 chars


## Fit the bigram model on train (n=2)

In [4]:
model = NGramModel.fit(train_names, vocab, n=2)
model.counts.shape

torch.Size([38, 38])

## Count matrix (bigram, n=2)


In [5]:
N = model.counts
chars = [vocab.itos[i] for i in range(len(vocab))]
fig = px.imshow(
    N.numpy(),
    x=chars,
    y=chars,
    text_auto=".0f",
    color_continuous_scale="Blues",
    title="bigram counts",
)
fig.update_layout(width=800, height=800)
fig.show()

## Sample names (n=2, min 3 chars)

In [ ]:
for name in model.sample(n=10, seed=135798):
    print(name)

## Evaluation — train / val (test kept aside)

In [ ]:
print("bigram (n=2) fit on train:")
print(f"  train NLL {model.nll(train_names):.4f} | val NLL {model.nll(val_names):.4f}")

## Orders compared — fit on train, choose n on val

Watch the val NLL as n grows: past n=3 the count tensor gets too sparse and
the smoothing floor dominates. The test split is only touched once, at the end.

In [ ]:
for n in (2, 3, 4):
    m = NGramModel.fit(train_names, vocab, n=n)
    print(f"n={n}: train NLL {m.nll(train_names):.4f} | val NLL {m.nll(val_names):.4f}")

m3 = NGramModel.fit(train_names, vocab, n=3)
print("samples (n=3):")
for name in m3.sample(n=5, seed=135798):
    print(name)

## Final evaluation on test

Run once, after choosing n on val — the test NLL is the honest score.

In [ ]:
# n=3 chosen on val above; evaluate it once on test
m3 = NGramModel.fit(train_names, vocab, n=3)
print(
    f"n=3: train NLL {m3.nll(train_names):.4f} | val NLL {m3.nll(val_names):.4f}"
    f" | test NLL {m3.nll(test_names):.4f}"
)